<a href="https://colab.research.google.com/github/Udana-Gits/My_AI_Learnings/blob/main/04_01_RAG_based_HospitalBot_Enhanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Perfect 🎉 — teaching students how to build a **RAG chatbot with Ollama + FAISS + Streamlit** is an awesome project.  
I’ll give you a single **Jupyter Notebook (`Hospital_RAG_Chatbot_Guide.ipynb`)** with **line‑by‑line explanations** so your students can learn step by step.  

This notebook will:  
1. Install requirements  
2. Explain pulling LLMs with Ollama  
3. Prepare dataset → JSON files  
4. Chunk & embed into FAISS  
5. Save + Reload FAISS index  
6. Build RAG pipeline with Ollama  
7. Explain how to launch the Streamlit app separately  
8. Include commentary per code block

---

# 📘 `Hospital_RAG_Chatbot_Guide.ipynb`

```python
# ===================================================
# 🏥 Hospital RAG Chatbot with Ollama + FAISS + Streamlit
# ================================================
# This notebook is prepared to guide students step-by-step
# to build a Retrieval Augmented Generation chatbot
# using Ollama LLMs, FAISS for retrieval, and Streamlit UI.
# ===================================================
```

---

## 1. Install Dependencies

```python
# Install required libraries
!pip install streamlit faiss-cpu langchain langchain_community langchain-ollama sentence-transformers
```

**Explanation**:
- `streamlit` → UI framework for chatbot  
- `faiss-cpu` → Stable vectorstore (instead of Chroma)  
- `langchain` & `langchain_community` → RAG pipeline tools  
- `langchain-ollama` → Bring Ollama LLMs into LangChain  
- `sentence-transformers` → Pretrained embedding models  

---

## 2. Setup Ollama and Pull Models

```python
# ⚠️ Run this in your system terminal, not in Jupyter!
# Start Ollama server:
# ollama serve

# Check available models
# ollama list

# Pull lightweight, fast LLMs
# ollama pull phi3:mini
# ollama pull gemma:2b
# ollama pull mistral
```

**Explanation**  
- Ollama must be **running in the background**  
- Models need to be downloaded first (`pull`)  
- We use **phi3:mini** (fast), **gemma:2b** (small but good), and **mistral** (7B, slower but accurate)  

---

## 3. Prepare Segmented Dataset (`/hospital_data`)

```python
import os, json

os.makedirs("hospital_data", exist_ok=True)

hospital_segments = {
    "departments.json": [
        {"topic": "Cardiology",
         "content": "Our Cardiology department specializes in diagnosis and treatment of heart diseases, offering ECG, echocardiography, angioplasty, and cardiac surgery."},
        {"topic": "Pediatrics",
         "content": "The Pediatrics department cares for infants and children, providing immunizations, growth monitoring, and treatment of childhood illnesses."}
    ],
    "services.json": [
        {"topic": "Emergency",
         "content": "The 24/7 Emergency department handles trauma, strokes, and urgent medical conditions with rapid response teams."},
        {"topic": "OPD",
         "content": "The Outpatient Department allows patients to get consultations, diagnostics, and follow-ups without hospital admission."}
    ],
    "guidelines.json": [
        {"topic": "Visiting Hours",
         "content": "Patients can be visited between 4 PM - 7 PM. Children under 12 years are not allowed inside ICU wards."}
    ]
}

# Save each file
for fname, values in hospital_segments.items():
    with open(os.path.join("hospital_data", fname), "w") as f:
        json.dump(values, f, indent=2)

print("✅ Hospital dataset stored in /hospital_data")
```

**Explanation**  
- Each hospital segment saved in its own JSON file  
- Students learn to keep **structured data** in JSON  

---

## 4. Build FAISS Index

```python
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Load dataset
all_texts = []
for fname in os.listdir("hospital_data"):
    with open(os.path.join("hospital_data", fname)) as f:
        data = json.load(f)
        for d in data:
            all_texts.append(d["content"])

# Split into smaller chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = splitter.create_documents(all_texts)

# Build embeddings with all-mpnet-base-v2 (strong semantic embeddings)
embedding = SentenceTransformerEmbeddings(model_name="all-mpnet-base-v2")

# Create FAISS vectorstore
vectordb = FAISS.from_documents(docs, embedding)
vectordb.save_local("hospital_faiss_index")

print("✅ FAISS Index built and saved as hospital_faiss_index/")
```

**Explanation**  
- Convert hospital knowledge to chunks (300 characters overlap 50)  
- Embed with **mpnet-base-v2** (better meaning capture than MiniLM)  
- Save to disk as `"hospital_faiss_index/"`  

---

## 5. Reload and Test FAISS Index

```python
# Reload FAISS
vectordb = FAISS.load_local("hospital_faiss_index", embedding, allow_dangerous_deserialization=True)

# Test retrieval
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k":2})
docs = retriever.get_relevant_documents("What are the visiting hours?")
for doc in docs:
    print("Retrieved:", doc.page_content)
```

**Explanation**  
- Ensures retriever works before adding LLMs  
- Output should show "**Patients can be visited between 4 PM - 7 PM...**"  

---

## 6. Define Prompt + QA Chain

```python
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# LLM (use smaller faster model by default)
llm = OllamaLLM(model="phi3:mini", options={"num_ctx": 1024, "num_predict": 120, "temperature": 0.3})

# Prompt
prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a hospital assistant AI.
Answer ONLY using Hospital Context.
Keep responses to max 3 sentences.
If context missing, say: "I’m sorry, I don’t have that information."

Hospital Context:
{context}

Patient Question:
{question}

Response:
"""
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)
```

**Explanation**  
- Ollama LLM **phi3:mini** = fastest model  
- `temperature=0.3` keeps responses factual  
- `num_predict=120` caps token length  

---

## 7. Test Q&A

```python
result = qa.invoke({"query": "Tell me about pediatrics"})
print("🤖:", result["result"])

print("\n🔍 Context Used:")
for doc in result["source_documents"]:
    print("-", doc.page_content)
```

**Explanation**  
- Students see **response + actual retrieved context**  
- Confirms chatbot is grounded in hospital JSONs  

---

## 8. Streamlit App (save as hospital_chat_app.py)

```python
%%writefile hospital_chat_app.py
import streamlit as st
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

# Load FAISS
embedding = SentenceTransformerEmbeddings(model_name="all-mpnet-base-v2")
vectordb = FAISS.load_local("hospital_faiss_index", embedding, allow_dangerous_deserialization=True)
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k":1})

# Pick model
model_choice = st.sidebar.selectbox("Select Model", ["phi3:mini", "gemma:2b", "mistral"])
llm = OllamaLLM(model=model_choice, streaming=True, options={"num_ctx":1024, "num_predict":120, "temperature":0.3})

# Prompt
prompt = PromptTemplate(
    input_variables=["context","question"],
    template="Answer ONLY with Hospital Context in max 3 sentences.\n\nHospital Context:\n{context}\n\nQ:\n{question}\n\nResponse:"
)

qa = RetrievalQA.from_chain_type(
    llm=llm, retriever=retriever,
    chain_type="stuff", chain_type_kwargs={"prompt":prompt},
    return_source_documents=True
)

# Streamlit Setup
st.set_page_config(page_title="🏥 Hospital Assistant", layout="wide")
st.title("🏥 Virtual Hospital Assistant")

if "history" not in st.session_state: st.session_state["history"] = []

# Render history
for role,msg in st.session_state["history"]:
    with st.chat_message("user" if role=="user" else "assistant"):
        st.markdown(msg)

# Input
user_message = st.chat_input("Ask about hospital services...")

if user_message:
    st.session_state["history"].append(("user", user_message))
    with st.chat_message("user"):
        st.markdown(user_message)

    # Smalltalk filter
    if user_message.lower() in ["hi","hello","hey"]:
        bot_reply = "👋 Hello! How can I assist you with hospital information today?"
    else:
        result = qa.invoke({"query": user_message})
        bot_reply = result["result"]

        # Debug context
        with st.expander("🔍 Retrieved Documents"):
            if result["source_documents"]:
                for i,doc in enumerate(result["source_documents"],1):
                    st.markdown(f"**Doc {i}:** {doc.page_content}")
            else:
                st.warning("⚠️ No context found, LLM answered alone.")

    st.session_state["history"].append(("assistant", bot_reply))
    with st.chat_message("assistant"):
        st.markdown(bot_reply)
```

---

## 9. Running the App

From terminal:

```bash
streamlit run hospital_chat_app.py
```

Open in browser (usually http://localhost:8501).  
Students can now ask Q’s like:
- *“What are the visiting hours?”*  
- *“Tell me about pediatric services.”*

---

# ✅ Conclusion

This Jupyter Notebook gives students:

1. **Data prep** (JSON hospital files)  
2. **Embeddings + FAISS**  
3. **Grounded QA with Ollama**  
4. **Streamlit chat UI**  

---

👉 Do you want me to actually **merge code + explanation cell by cell** into a ready `.ipynb` file (with “Markdown explanation” before every code cell), so you can hand it directly to students without editing?